# Week 10 Live Coding
## Grading a turnout model

Keystone Analytics sold us a turnout score for all 60,000 registered voters in the district. We have the same voter file they used, so we can grade it.

Four things we will do:
1. Grade the file Keystone sent
2. Build our own model and read its coefficients
3. Check calibration, twice
4. Build the walk list

*(The file is synthetic. It was built for this class so that every number on the slides is exactly reproducible.)*

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the voter file straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Run the cell below. The file is 60,000 rows, so give it a couple of seconds.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

vf = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk10_turnout_models/data/district_voter_file.csv')
print(vf.shape)
vf.head()

## Part 1: What did we buy?

One row per registered voter. Seven columns of vote history, three of demographics, and `keystone_score`.

Start with the thing every vendor file should be checked against first: what actually happened.

In [ ]:
# Turnout on this file, election by election
for year in [2012, 2014, 2016, 2018, 2020, 2022, 2024]:
    print(year, round(100 * vf['voted_' + str(year)].mean(), 1), '%')

Presidential years run from 57% to 76%, and they have been climbing. Midterms run around 41–51%. Hold on to that.

### Holding voters back

We are about to build our own model. If we grade it on the same people we built it from, it can look good by memorizing them. So set aside a random 18,000 voters now and use them for every number from here on.

(We cannot do this for Keystone. Their model has already seen all 60,000. That can only flatter them, which makes anything bad we find conservative.)

In [ ]:
train = vf.sample(frac=0.7, random_state=10)
test = vf.drop(train.index).copy()
print(len(train), 'to build on;', len(test), 'held back')

### The accuracy floor

Keystone's pitch: *validated against the 2024 general election, the model classified 80.0% of registered voters correctly.*

Call anyone scored at 0.5 or above a voter, anyone below a non-voter, and compare to 2024.

Before you read the number, work out what the dumbest possible model scores. It writes the **same answer on every row** — whichever answer is more common.

In [ ]:
keystone_call = (test['keystone_score'] >= 0.5).astype(int)
accuracy = (keystone_call == test['voted_2024']).mean()

turnout_2024 = test['voted_2024'].mean()
floor_2024 = max(turnout_2024, 1 - turnout_2024)   # the majority answer

print('Keystone accuracy:', round(100 * accuracy, 1), '%')
print('Writing the same answer on every row:', round(100 * floor_2024, 1), '%')
print('Work done:', round(100 * (accuracy - floor_2024), 1), 'points')

Four points. Keystone's headline number is mostly the turnout rate in a presidential year.

Note the floor is the **majority** answer, not always "will vote." Turnout in 2014 was 41.1%, so there the free answer is "will not vote" and it scores 58.9%.

## Part 2: Build our own

Same idea, one line of code. Predict turnout from the four general elections before 2020, plus age.

We will fit it twice with **the same right-hand side**, changing only which election is on the left: 2024, a presidential, and 2022, a midterm.

In [ ]:
X = 'voted_2012 + voted_2014 + voted_2016 + voted_2018 + age'

pres = smf.ols('voted_2024 ~ ' + X, data=train).fit()
mid = smf.ols('voted_2022 ~ ' + X, data=train).fit()

pd.DataFrame({'fit to 2024 (presidential)': (100 * pres.params).round(1),
              'fit to 2022 (midterm)': (100 * mid.params).round(1)})

Multiply by 100 and a coefficient reads in percentage points, same as Week 2.

**2014 and 2018 were midterms. 2012 and 2016 were presidentials.**

To predict a midterm, having voted in a past midterm is worth 27.7 and 33.1 points. Having voted in a past presidential is worth 10.4 and 11.8. To predict a presidential it hardly matters which.

The intercept moves from 58.8 to 14.5. These are not the same model with a different starting point. The weights on the columns moved too.

Here is the midterm model in full.

In [ ]:
mid.summary()

Four things to look at and the rest to ignore: the coefficient, its standard error, and (from the header) $N$ and $R^2$.

Notice the standard error on `age` prints as `0.000`. Published tables round, and rounding can hide a number you care about.

### A linear probability model

We fit a straight line to a 0/1 outcome. That is a **linear probability model**, and it is why we can read the coefficients in points. It has one flaw: a straight line does not know a probability has to stay between 0 and 1.

In [ ]:
test['our_score'] = mid.predict(test)
raw_pres = pres.predict(test)

print('our midterm scores :', round(test['our_score'].min(), 2), 'to', round(test['our_score'].max(), 2))
print('the presidential fit:', round(raw_pres.min(), 2), 'to', round(raw_pres.max(), 2))
print('  of those,', round(100 * (raw_pres > 1).mean(), 1), '% are above 1.00')

Midterm turnout is near 50%, in the middle of the range, so the line has room. In a presidential year it does not.

The fix we use is to **trim at 0 and 1**. Keystone's file already comes that way, which is why its scores stop at exactly 1.00 and 9,595 people are tied there.

Now score the held-out voters and grade the midterm model.

In [ ]:
our_call = (test['our_score'] >= 0.5).astype(int)
accuracy = (our_call == test['voted_2022']).mean()

turnout_2022 = test['voted_2022'].mean()
floor_2022 = max(turnout_2022, 1 - turnout_2022)

print('Our accuracy:', round(100 * accuracy, 1), '%')
print('The free answer:', round(100 * floor_2022, 1), '%')
print('Work done:', round(100 * (accuracy - floor_2022), 1), 'points')

Almost exactly the same accuracy as Keystone, doing seven times as much work.

That difference is about the two **elections**, not about the two models. Fit our own model to 2024 and it scores 80.0% against a floor of 75.9%, exactly like Keystone.

### Which mistakes?

Accuracy treats a false positive and a false negative as the same error. They are not. A false positive is a door you did not need to knock. A false negative is someone you wrote off.

In [ ]:
confusion = pd.crosstab(our_call, test['voted_2022'])
confusion.index = ['model said will not vote', 'model said will vote']
confusion.columns = ['did not vote', 'voted']
confusion

In [ ]:
tp = ((our_call == 1) & (test['voted_2022'] == 1)).sum()
fp = ((our_call == 1) & (test['voted_2022'] == 0)).sum()
fn = ((our_call == 0) & (test['voted_2022'] == 1)).sum()

print('precision:', round(100 * tp / (tp + fp), 1), '%   of those we called voters, this share voted')
print('recall:   ', round(100 * tp / (tp + fn), 1), '%   of the people who voted, we called this share')
print()
print('precision floor (call everyone a voter):', round(100 * turnout_2022, 1), '%')
print('recall floor    (call everyone a voter): 100.0 %')

Compare that to Keystone's recall against 2024, which is 95.2%. It looks wonderful, and it is bought by calling 15,956 of 18,000 people voters. Recall is easy to buy: say yes to everyone and it is 100%.

## Part 3: Calibration

Accuracy, precision and recall all throw the number away and keep a yes or a no. Calibration keeps the number, and the number is what any decision with a *score cutoff* uses.

**A model is calibrated when its numbers mean what they say: of the people it scores 0.60, about 60% vote.**

We will also want one summary number. The **Brier score** is the average of (score − outcome)², where the outcome is 1 if they voted and 0 if they did not. Lower is better.

In [ ]:
def calibration(scores, actual):
    # bins must reach past 1.0 in case any score does; pd.cut silently drops
    # anything outside its edges.
    top = max(1.0, np.ceil(scores.max() * 10) / 10)
    bins = pd.cut(scores, np.arange(0, top + 0.001, 0.1), include_lowest=True)
    out = pd.DataFrame({'score': scores, 'voted': actual}).groupby(bins, observed=True).agg(
        n=('voted', 'size'), model_said=('score', 'mean'), actually_voted=('voted', 'mean'))
    out['gap_pts'] = (out['model_said'] - out['actually_voted']) * 100
    return out.round(3)

brier = lambda scores, actual: round(float(((scores - actual) ** 2).mean()), 4)

print('our model, Brier:', brier(test['our_score'], test['voted_2022']))
print('giving everyone the same turnout rate:', brier(train['voted_2022'].mean(), test['voted_2022']))
calibration(test['our_score'], test['voted_2022'])

The dots sit on the line. The worst bin is the bottom one, off by about four points.

Now the question this week is really about. Keystone fit their model to the **2024 general**, a presidential. We are running in **November 2026**, a midterm.

We cannot grade a prediction of 2026. But 2022 was a midterm, it already happened, and it is in the file. So grade Keystone's scores against 2022.

In [ ]:
print('Keystone, Brier against 2022:', brier(test['keystone_score'], test['voted_2022']))
print('mean Keystone score:', round(100 * test['keystone_score'].mean(), 1), '%',
      ' vs actual turnout:', round(100 * test['voted_2022'].mean(), 1), '%')
calibration(test['keystone_score'], test['voted_2022'])

Every single bin over-predicts. Look at the `gap_pts` column: the smallest miss is about 9 points and the largest is 42.

Its Brier score against a midterm is 0.217, against 0.136 for the model we just built on the same people.

Let's look at the two side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharey=True)

for ax, col, title in [(axes[0], 'our_score', 'Our model, fit to a midterm'),
                       (axes[1], 'keystone_score', 'Keystone, fit to a presidential')]:
    tab = calibration(test[col], test['voted_2022'])
    ax.plot([0, 1], [0, 1], color='#555555', ls='--', lw=1)
    ax.scatter(tab['model_said'], tab['actually_voted'], s=tab['n'] / 9,
               color='#0F4D92' if col == 'our_score' else '#B58900')
    ax.set_title(title)
    ax.set_xlabel('What the model said')
    ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)

axes[0].set_ylabel('Share who actually voted in 2022')
plt.tight_layout()
plt.show()

### So whose list actually changes?

Here is the thing that surprises people. The two models put people in **almost the same order** — so if all you do is take the top of the list, Keystone's file is fine.

What breaks is any decision defined by a score *cutoff*, because Keystone's scores never go below 0.36.

In [ ]:
vf['score'] = mid.predict(vf)
print('rank correlation:', round(vf[['keystone_score', 'score']].corr(method='spearman').iloc[0, 1], 3))
top_k = set(vf.nlargest(15000, 'keystone_score').index) & set(vf.nlargest(15000, 'score').index)
print('the top 15,000 by each score share', len(top_k), 'people')
print()
print('Keystone scores run', round(vf['keystone_score'].min(), 2), 'to', round(vf['keystone_score'].max(), 2))
print('ours run           ', round(vf['score'].min(), 2), 'to', round(vf['score'].max(), 2))

In [ ]:
# Katie wants everyone between 0.35 and 0.65. Whose scores?
katie_ours = vf[vf['score'].between(0.35, 0.65)]
katie_keys = vf[vf['keystone_score'].between(0.35, 0.65)]

for name, rows in [('by our midterm model', katie_ours), ('by Keystone', katie_keys)]:
    print(name.ljust(22), len(rows), 'people |',
          str(round(100 * rows['voted_2022'].mean(), 1)) + '% of them voted in 2022')
print('in both lists:', len(set(katie_ours.index) & set(katie_keys.index)))

Twenty-two thousand people in Keystone's "middle," and 14% of them voted. Ten thousand in ours, and 57% did. They share 677 people.

Rachel's decision does not need a good model. Katie's decision **is** the model.

## Part 4: The walk list

Sort all 60,000 into three bands and see what each of the three proposed lists would have bought.

In [ ]:
vf['band'] = pd.cut(vf['score'], [-1, 0.35, 0.65, 2], labels=['low', 'middle', 'high'])

vf.groupby('band', observed=True).agg(n=('score', 'size'),
                                      mean_score=('score', 'mean'),
                                      voted_2022=('voted_2022', 'mean')).round(3)

The mean score in each band lands within about a point of what those people actually did in 2022.

Now the three lists our office is arguing about. 9,000 doors each.

In [ ]:
for name, rows in [('Rachel: 9,000 highest', vf.nlargest(9000, 'score')),
                   ('Katie: 9,000 in the middle', vf[vf['band'] == 'middle'].sample(9000, random_state=3)),
                   ('Marcus: 9,000 lowest', vf.nsmallest(9000, 'score'))]:
    print(name.ljust(28),
          'mean score', format(rows['score'].mean(), '.2f'),
          '| voted in 2022:', str(round(100 * rows['voted_2022'].mean(), 1)) + '%')

Rachel would spend 90% of the volunteer program on people who were coming anyway. Marcus would spend it on people who did not vote in 2022 at all.

One more constraint before you write a walk list.

In [ ]:
counts = pd.crosstab(vf['band'], vf['party'])
print(counts)
print()
mid_du = counts.loc['middle', 'D'] + counts.loc['middle', 'U']
print('middle band, registered D or unaffiliated:', mid_du)
print('doors we can knock:', 9000)

Katie's band cannot fill 9,000 doors once you drop the Republicans. You would have to widen it.

And party **registration** is not support: 30.8% of this district is unaffiliated, and a turnout model says nothing about who anyone supports.

The deeper problem is bigger than that. Every number in this notebook is about the probability someone votes **if you do nothing**. What you actually need is how much your knock **changes** that. Nothing in a voter file contains it, because a voter file has no experiment in it.

That is the whole course, arriving in a column of scores.